In [ ]:
!pip install -U qwen-tts

In [ ]:
import torch
import os

from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel

model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
model = Qwen3TTSModel.from_pretrained(model_id, device_map=device, dtype=dtype)

prompt_file_path = "./voice-data/voice_prompt.pt" # 저장할 파일 이름 확장자는 보통 .pt 나 .pth를 씁니다.

In [1]:
import gradio as gr
import torch
import soundfile as sf
import numpy as np

from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel

# 1. 현재 환경 자동 감지 (Colab vs Local)
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# 2. 하드웨어 가속 및 데이터 타입 자동 설정
if torch.cuda.is_available():
    device = "cuda:0"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print(f"🌍 현재 환경: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"💻 로드된 디바이스: {device} (dtype: {dtype})")

# 3. 모델 로드
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"
model = Qwen3TTSModel.from_pretrained(model_id, device_map=device, dtype=dtype)

def generate_audio(text, lang):
    output_path = "temp_audio.wav"
    speaker = "Sohee"
    wavs, sr = model.generate_custom_voice(
        text=text,
        language=lang,
        speaker=speaker,
    )

    sf.write(output_path, wavs[0], sr)

    return output_path

# Gradio 인터페이스 구성
demo = gr.Interface(
    fn=generate_audio,
    inputs=[
        gr.Textbox(label="Text (합성할 텍스트)", placeholder="여기에 대본을 넣으세요..."),
        gr.Dropdown(label="Language", choices=["Korean", "Chinese", "English"], value="Korean")
    ],
    outputs=gr.Audio(label="Output Audio"),
    title="Qwen3-TTS Custom Voice Studio (Colab Edition)"
)

# 5. 환경에 따른 실행 옵션 자동 분기
if IS_COLAB:
    # 코랩: 외부 브라우저에서 접속하기 위해 share=True (터널링) 필수
    print("🚀 코랩 환경이 감지되었습니다. Public URL을 생성합니다.")
    demo.launch(share=True, debug=True)
else:
    # 로컬: 8000번 포트 고정, 내부 네트워크에서만 접근 (보안 유지 및 속도 빠름)
    print("🏠 로컬 환경이 감지되었습니다. http://localhost:8000 으로 접속하세요.")
    demo.launch(server_name="0.0.0.0", server_port=8000, share=False, debug=True)

/Users/kdm/miniconda3/envs/qwen3-tts/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



********
********
 
🌍 현재 환경: Local
💻 로드된 디바이스: mps (dtype: torch.float16)


Fetching 4 files: 100%|██████████| 4/4 [00:11<00:00,  2.92s/it]


🏠 로컬 환경이 감지되었습니다. http://localhost:8000 으로 접속하세요.
* Running on local URL:  http://0.0.0.0:8000
* To create a public link, set `share=True` in `launch()`.


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Keyboard interruption in main thread... closing server.


In [ ]:
import gradio as gr
import torch
import soundfile as sf
import numpy as np
import os

from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel

# 1. 현재 환경 자동 감지 (Colab vs Local)
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# 2. 하드웨어 가속 및 데이터 타입 자동 설정
if torch.cuda.is_available():
    device = "cuda:0"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print(f"🌍 현재 환경: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"💻 로드된 디바이스: {device} (dtype: {dtype})")

# 3. 모델 로드
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
model = Qwen3TTSModel.from_pretrained(model_id, device_map=device, dtype=dtype)
prompt_file_path = "./voice-data/voice_prompt.pt"

# 4. Voice 데이터 가져오기
if os.path.exists(prompt_file_path):
    # 파일이 있으면 저장된 목소리 특징 불러오기
    print("저장된 목소리 특징 파일을 불러옵니다...")
    prompt_items = torch.load(prompt_file_path, map_location=device) 
else:
    # 파일이 없으면 새로 추출하고 파일로 저장하기
    print("목소리 특징 파일이 없습니다. 새로 추출을 시작합니다...")
    prompt_items = model.create_voice_clone_prompt(
        ref_audio="./audio-ref/dm_audio.wav",
        ref_text="안녕하세요! 오늘은 최근 AI 업계에서 가장 뜨거운 감자인 '하네스 엔지니어링'에 대해 알아보려고 합니다. 단순한 코딩을 넘어 시스템 전체를 설계하는 이 새로운 개념이 과연 우리의 미래를 어떻게 바꿀까요? 지금 바로 시작합니다!",
        x_vector_only_mode=False,
    )
    # 추출한 특징을 파일로 영구 저장
    torch.save(prompt_items, prompt_file_path)
    print(f"목소리 특징이 {prompt_file_path}에 저장되었습니다!")

def generate_audio(text, lang):
    output_path = "./temp_audio.wav"

    wavs, sr = model.generate_voice_clone(
        text=text,
        language=lang,
        voice_clone_prompt=prompt_items
    )

    wav = wavs[0]
    silence = np.zeros(int(sr * 0.5), dtype=wav.dtype)
    wav = np.concatenate([wav, silence])

    sf.write(output_path, wav, sr)

    return output_path

# 4. Gradio 인터페이스 구성
demo = gr.Interface(
    fn=generate_audio,
    inputs=[
        gr.Textbox(label="Text (합성할 텍스트)", placeholder="여기에 대본을 넣으세요..."),
        gr.Dropdown(label="Language", choices=["Korean", "Chinese", "English"], value="Korean")
    ],
    outputs=gr.Audio(label="Output Audio"),
    title=f"Qwen3-TTS Voice Design Studio ({'Colab' if IS_COLAB else 'Local'} Edition)"
)

# 5. 환경에 따른 실행 옵션 자동 분기
if IS_COLAB:
    # 코랩: 외부 브라우저에서 접속하기 위해 share=True (터널링) 필수
    print("🚀 코랩 환경이 감지되었습니다. Public URL을 생성합니다.")
    demo.launch(share=True, debug=True)
else:
    # 로컬: 8000번 포트 고정, 내부 네트워크에서만 접근 (보안 유지 및 속도 빠름)
    print("🏠 로컬 환경이 감지되었습니다. http://localhost:8000 으로 접속하세요.")
    demo.launch(server_name="0.0.0.0", server_port=8000, share=False, debug=True)

🌍 현재 환경: Local
💻 로드된 디바이스: mps (dtype: torch.float16)


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 13562.83it/s]


NameError: name 'os' is not defined